# Can we predict 30-day readmission?
Methodology follows `checklist.md`. Key facts about the data: 11,868 stays from 1,500 people (about 8 stays each), and no calendar date column. Rows are therefore **not independent**, so every split (test set and CV folds) keeps whole people together. `visit_number` is a per-person sequence, not a date, so a "latest rows" split isn't possible.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
from sklearn.base import clone
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, df.person_key.nunique(), 'prevalence', df.readmit_30d.mean().round(3))

## 1. Feature audit (checklist 7)
Prediction time is **discharge** (the outcome is readmission after discharge). `columns.csv` shows all remaining columns are known by then: admission labs/BMI, stay length and discharge medication count. `person_key` is an identifier and is dropped (used only for grouping). `postcode_area` is a numeric code of an area, so it is treated as categorical.

In [ ]:
ID='person_key'; TARGET='readmit_30d'
NUM=['visit_number','birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
CAT=['sex','postcode_area']
X=df[NUM+CAT].copy(); X['postcode_area']=X['postcode_area'].astype(str); y=df[TARGET].values; g=df[ID].values

## 2. Test set held out first, by whole person (checklist 1)

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
tr,te=next(gss.split(X,y,g))
Xtr,ytr,gtr=X.iloc[tr],y[tr],g[tr]; Xte,yte=X.iloc[te],y[te]
assert not set(g[tr])&set(g[te])
print(len(tr),len(te),ytr.mean().round(3),yte.mean().round(3))

## 3. Candidates, declared before any scoring (checklist 4)
1. `baseline`: predicts training prevalence (trivial)
2. `logreg`: logistic regression, scaled numerics + one-hot categoricals; C in {0.01, 0.1, 1}
3. `rf`: random forest; min_samples_leaf in {5, 20}
4. `hgb`: histogram gradient boosting; max_depth in {2, 3}, learning_rate 0.05

Selection metric: out-of-fold ROC AUC (PR AUC reported alongside). Features: all columns audited above. Decision threshold: chosen on pooled out-of-fold predictions of the winner, maximising F1. Steps are fixed here; the test set is not touched until section 5.

In [ ]:
def prep(scale):
    return ColumnTransformer([('n',StandardScaler() if scale else 'passthrough',NUM),
                              ('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),CAT)])
def pipe(est,scale=False): return Pipeline([('prep',prep(scale)),('m',est)])
cands={'baseline':pipe(DummyClassifier(strategy='prior'))}
for C in [0.01,0.1,1]: cands[f'logreg C={C}']=pipe(LogisticRegression(C=C,max_iter=2000),True)
for l in [5,20]: cands[f'rf leaf={l}']=pipe(RandomForestClassifier(300,min_samples_leaf=l,n_jobs=-1,random_state=SEED))
for d in [2,3]: cands[f'hgb depth={d}']=pipe(HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=200,random_state=SEED))

## 4. Grouped cross-validation on training rows only (checklist 2, 3, 5)
Preprocessing lives inside each pipeline, so it is refit in every fold.

In [ ]:
cv=GroupKFold(n_splits=5)
folds=list(cv.split(Xtr,ytr,gtr))
oof={}; rows=[]
for name,p in cands.items():
    o=np.zeros(len(ytr)); fa=[]
    for a,b in folds:
        m=clone(p).fit(Xtr.iloc[a],ytr[a]); o[b]=m.predict_proba(Xtr.iloc[b])[:,1]
        fa.append(roc_auc_score(ytr[b],o[b]))
    oof[name]=o
    rows.append((name,np.mean(fa),np.std(fa),average_precision_score(ytr,o)))
cvres=pd.DataFrame(rows,columns=['model','cv_auc_mean','cv_auc_sd','oof_pr_auc']).sort_values('cv_auc_mean',ascending=False)
cvres

In [ ]:
best=cvres.iloc[0].model
o=oof[best]; ths=np.linspace(0.05,0.6,111)
f1s=[f1_score(ytr,o>=t) for t in ths]; thr=float(ths[int(np.argmax(f1s))])
print('chosen:',best,'| threshold (max OOF F1):',round(thr,3),'| OOF F1:',round(max(f1s),3),
      '| F1 if flag everyone:',round(f1_score(ytr,np.ones_like(ytr)),3))

## 5. Final score, once (checklist 6)
The choice above (model, hyperparameters, threshold) is fixed from CV. Refit on all training rows and score the test set a single time.

In [ ]:
final=clone(cands[best]).fit(Xtr,ytr)
p=final.predict_proba(Xte)[:,1]
rng=np.random.default_rng(SEED); gte=g[te]; ug=np.unique(gte); idx={u:np.where(gte==u)[0] for u in ug}
bs=[]
for _ in range(500):  # person-level bootstrap for the AUC interval
    s=np.concatenate([idx[u] for u in rng.choice(ug,len(ug))]); 
    if len(set(yte[s]))>1: bs.append(roc_auc_score(yte[s],p[s]))
print(f'TEST ROC AUC {roc_auc_score(yte,p):.3f}  (95% CI {np.percentile(bs,2.5):.3f}-{np.percentile(bs,97.5):.3f})')
print(f'TEST PR AUC  {average_precision_score(yte,p):.3f}  (prevalence {yte.mean():.3f})')
print(f'TEST F1 at thr {thr:.2f}: {f1_score(yte,p>=thr):.3f}   Brier {brier_score_loss(yte,p):.3f}')

## Conclusion
Results from running these cells (seed 0): CV chose `logreg C=0.01` (out-of-fold AUC 0.642 ± 0.013 vs 0.5 for the baseline). Single test-set score: **ROC AUC 0.655 (95% person-level bootstrap CI 0.626–0.684)**, PR AUC 0.335 against 0.199 prevalence, F1 0.386 at the CV-chosen threshold 0.21 (flagging everyone gives about 0.33).

So readmission is predictable only weakly: better than chance, but not good enough to rely on for individual decisions. Caveats: no dates, so we tested generalisation to unseen people, not to later periods; the candidate set was small; the numbers above were pasted from a script run of the same code, so re-run the notebook to regenerate outputs.